In [ ]:
import sys
sys.path.append('../../Scripts')

%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import os
import subprocess as sub
import py3Dmol
import dill
import numpy as np
from sklearn.manifold import MDS
import umap
import more_itertools as mit
from scipy.signal import find_peaks
import meeko

from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.cluster import KMeans
from rdkit.Chem import rdMolTransforms
from rdkit.Chem.rdMolAlign import AlignMolConformers
from rdkit.Chem import rdDistGeom

from pathlib import Path
from vina import Vina

# User defined functions
sys.path.append('../python')
from Molecule import Molecule
from MolViz3D import MolViz3D
from RDKit_tools import clustering_coords

# FUNCTIONS

In [ ]:
def create_multiple_confs_from_smiles(smiles, out_path='', n_confs=100, n_clusters=10, how='etkdg', v=1):
    """
    From smiles generates multiple conformers.
    :param str smiles: the smiles string
    :param str out_path: path to write .pdb
    :return list[Molecule()]: list of conformations with correct glutarimide
    """
    ## find whether glutarimide or DHU
    mol = Chem.MolFromSmiles(smiles)

    ## generate multiple conformers:
    m, c_ids = clustering_coords(mol, M=n_confs, N=n_clusters, how=how)

    ## convert Molecules:
    opt_confs = []
    for c in c_ids:

        ## extract coordinates of molecules
        _coords = m.GetConformers()[c].GetPositions().astype(np.float32)  # change coordinates based on algo

        processed_molblock = Chem.SDWriter.GetText(m)
        sdf_opt = Molecule()
        sdf_opt.import_sdf(processed_molblock, from_text=True)
        sdf_opt.set_xyz(_coords)
        sdf_opt = sdf_opt.renumber_atomname()

        opt_confs.append(sdf_opt)

    return opt_confs

In [ ]:
## to be moved somewhere else
def prepare_ligand_pdbqt_from_smiles(smiles, single=False):
    """
    As the name suggest does all the preparation for you
    from the smiles string
    :return str pqbdt
    """
    # Convert SMILES to 3D structure
    if single:
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.UFFOptimizeMolecule(mol)

        # prepare pdbqt from meeko
        preparator = meeko.MoleculePreparation()
        mol_setups = preparator.prepare(mol)
        pdbqts = [meeko.PDBQTWriterLegacy.write_string(mol_setups[0])[0]]
    else:
        # find whether glutarimide or DHU
        mol = Chem.MolFromSmiles(smiles)

        # generate multiple conformers:
        m, c_ids = clustering_coords(mol, M=100, N=10, how='etkdg')

        # convert Molecules:
        pdbqts = []
        for c_id in c_ids:

            mol_conf = Chem.Mol(m)
            mol_conf.RemoveAllConformers()
            mol_conf.AddConformer(m.GetConformer(c_id), assignId=True)
            # print(Chem.SDWriter.GetText(mol_conf))

            mol_conf = Chem.AddHs(mol_conf)  # Add hydrogens

            # prepare pdbqt from meeko
            preparator = meeko.MoleculePreparation()
            mol_setups = preparator.prepare(mol_conf)
            # print(meeko.PDBQTWriterLegacy.write_string(mol_setups[0]))
            pdbqt = meeko.PDBQTWriterLegacy.write_string(mol_setups[0])[0]
            # print(pdbqt)

            pdbqts.append(pdbqt)

    return pdbqts

In [ ]:
def prepare_ligand_pdbqt_from_sdf(sdf_path):
    """
    Does what the name suggestes
    """
    # print(sdf_path)
    suppl = Chem.SDMolSupplier(sdf_path)
    mol = suppl[0]
    mol = Chem.AddHs(mol, addCoords=True)

    ## prepare pdbqt from meeko
    preparator = meeko.MoleculePreparation()
    mol_setups = preparator.prepare(mol)
    pdbqts = [meeko.PDBQTWriterLegacy.write_string(mol_setups[0])[0]]

    return pdbqts[0]

In [ ]:
def generate_gaussian_noise_vector(shape, mean=0.0, std=1.0):
    """
    Generates a random Gaussian noise vector.

    Args:
        shape: The shape of the output array (e.g., (100,), (10, 10)).
        mean: The mean of the Gaussian distribution.
        std: The standard deviation of the Gaussian distribution.

    Returns:
        A numpy array containing random Gaussian noise.
    """
    noise_vector = np.random.normal(loc=mean, scale=std, size=shape)
    return noise_vector

# MAIN

## 0. Imports

In [ ]:
## original data path
data_path = Path("../../structures/guidedock/").resolve()
receptor_paths = {f.parent.stem: f.resolve() for f in data_path.rglob("*_protein.pdb")}
crystal_ligand_paths = {f.parent.stem: f.resolve() for f in data_path.rglob("*_ligand.sdf")}
constraint_molecule_paths = crystal_ligand_paths
ligand_start_paths = {f.parent.stem: f.resolve() for f in data_path.rglob("*_GenConf_H.sdf")}

In [ ]:
ids = [x.split('/')[-1] for x in glob.glob('../../structures/guidedock/*')]

In [ ]:
%%time
## Docking via Autodock vina:
for tmp_id in ids:
    print('> docking', tmp_id)
    # tmp_id = 'alk1'
    ligand_sdf = ligand_start_paths[tmp_id].as_posix()
    ligand_pdbqt = prepare_ligand_pdbqt_from_dfs(ligand_sdf)

    ## Prepare protein pdbqt:
    prot = receptor_paths[tmp_id].as_posix()
    prot_pdbqt = prot + 'qt'

    if not os.path.exists(prot_pdbqt):
        ## write pdbqt of protein to disk:
        with open(os.devnull, 'wb') as shutup:
            sub.call('obabel -ipdb ' + prot + ' -O ' + prot_pdbqt + ' -p 7.4 -xr', shell=True, stderr=shutup)
    else:
        print(prot_pdbqt, 'found')

    # dock compound:
    # ---------------
    trial = 0

    # get crystal structure to set box parameters:
    xtal = Molecule()
    xtal.import_from_suffix(crystal_ligand_paths[tmp_id].as_posix())
    noise = generate_gaussian_noise_vector(3, 0.0, 1.0)

    # get pdqbt of ligand as text:
    lig_M = Molecule()
    lig_M.import_pdb(ligand_pdbqt, from_text=True)
    pdbtxt = lig_M.write_pdb()
    outpath = '/home/jovyan/work/guidedock/vina/' + tmp_id + '.pdbqt'

    # create pdbqt to extract for A. vina's optimization
    try:
        v = Vina(sf_name='vina', verbosity=0)
        v.set_receptor(prot_pdbqt)
        v.set_ligand_from_string(ligand_pdbqt)

        # set box conditions
        box_l = (xtal.rgyr() * 2.857) + 5.0
        v.compute_vina_maps(center=xtal.center() + noise, box_size=[box_l, box_l, box_l])
        # print('>> docking', outpath)
        # Dock the ligand
        v.dock(exhaustiveness=32, n_poses=10)
        v.write_poses(outpath, n_poses=10, overwrite=True)
    except:
        print('>> could not dock!')

## Visualization

In [ ]:
id = ids[0]  # Select an ID to visualize
prot = receptor_paths[id].as_posix()
ref = crystal_ligand_paths[id].as_posix()
# dock = res_dock[id].as_posix()  # Uncomment when res_dock is defined

In [ ]:
## Visualize predicted contacts on raw pharao
corecols = ['white', 'cyan', 'grey', 'orange', 'green', 'purple', 'blue', 'black', 'brown', 'magenta', 'cornflowerblue'] * 3
M3D = MolViz3D()

## ligands
M3D.show_ligand(ref, radius=0.05, colorscheme='whiteCarbon')  # _renum
# M3D.show_ligand(dock, radius=0.12, colorscheme='cyanCarbon')  # Uncomment when dock is defined

## protein
M3D.show_protein_single_color(prot, radius=None)  # style='sphere'
M3D.view.zoomTo(viewer=(100, 0))
M3D.view.show()